# NeuroWorkflow: E-I Network with Four Synaptic Receptors

A Brunel-style balanced network where every neuron carries **four receptors** instead of
one, so excitation and inhibition each arrive through a fast and a slow channel:

| receptor | number | tau_syn | role |
|---|---|---|---|
| AMPA   | 1 | 2 ms   | fast excitation |
| NMDA   | 2 | 100 ms | slow excitation |
| GABA_A | 3 | 6 ms   | fast inhibition |
| GABA_B | 4 | 200 ms | slow inhibition |

```
Drive (Poisson) ──AMPA──┐
                        ├─→ Connectivity → SimConfig → Analysis
PopExc ──AMPA, NMDA─────┤
PopInh ──GABA_A, GABA_B─┘
```

Excitatory cells contact their targets through two projections, AMPA and NMDA; inhibitory
cells through GABA_A and GABA_B. The external drive arrives on AMPA.

### What the model does and does not capture

`iaf_psc_alpha_multisynapse` is **current-based**: `tau_syn` is a list with one time
constant per receptor, and there are no reversal potentials. Two consequences worth
being explicit about:

- **A receptor sets the shape, the weight sets the sign.** Inhibition is a negative
  weight, not a reversal potential below rest. Inhibition here is hyperpolarizing
  regardless of membrane voltage, and cannot shunt.
- **NMDA has no magnesium block.** A real NMDA receptor conducts only once the cell is
  already depolarized. Here it is simply a slow excitatory current, open from the first
  spike. If you need the voltage dependence, that is a conductance-based model with an
  explicit Mg term, not this one.

### Charge, not weight, is what you are setting

An alpha-shaped PSC with peak `w` and time constant `tau` delivers charge proportional
to `w x tau`. So the weights below are not comparable across receptors:

| | weight | tau | w x tau |
|---|---|---|---|
| AMPA   | 40 | 2   | 80   |
| NMDA   | 2  | 100 | 200  |
| GABA_A | -150 | 6 | -900 |
| GABA_B | -8 | 200 | -1600 |

NMDA carries **2.5x the charge of AMPA at one twentieth the weight**. Section 5 shows
what happens when you forget that.

## 1. Instantiate nodes

In [ ]:
from neuroworkflow import WorkflowBuilder
from neuroworkflow.nodes.network.NW_Population    import NW_Population
from neuroworkflow.nodes.network.NW_Connectivity  import NW_Connectivity
from neuroworkflow.nodes.stimulus.NW_SpikeTrains  import NW_SpikeTrains
from neuroworkflow.nodes.simulation.NW_SimConfig  import NW_SimConfig
from neuroworkflow.nodes.analysis.NW_Analysis     import NW_Analysis

In [ ]:
exc   = NW_Population("PopExc")
inh   = NW_Population("PopInh")
drive = NW_SpikeTrains("Drive")
conn  = NW_Connectivity("Connectivity")
cfg   = NW_SimConfig("SimConfig")
ana   = NW_Analysis("Analysis")

## 2. Configure

Three things here are specific to multi-receptor networks and easy to get wrong.

**Receptors are numbered from 1**, in the order of `tau_syn`. `tau_syn[0]` is receptor 1.
A connection selects one with `receptor_type` inside `dynamics_params_dict`, which
NeuroWorkflow passes through to NEST's connection call. Omitting it on a multi-receptor
target fails with `IncompatibleReceptorType`; setting it on an ordinary single-receptor
model fails with `UnknownReceptorType`.

**Every projection needs its own `dynamics_params` filename.** The synapse parameters are
written to a JSON file named by that field, which defaults to the same name for every
connection. Two projections that both accept the default overwrite each other's file and
end up sharing one receptor - the network still runs, and is silently wrong.

**AMPA and NMDA must land on the same pairs.** They are two components of one synapse, not
two independent synapses. `np.random.rand() < 0.1`, the usual sparse rule, draws afresh
every time it is asked, so the two projections would be wired to different pairs. Seeding
a generator with the pathway number and the pair of node ids gives the same 10% sparse
connectivity while returning the same answer whenever it is asked the same question:

```python
"lambda src, tgt: 1 if np.random.default_rng([2, src['node_id'], tgt['node_id']]).random() < 0.1 else 0"
```

The rule is written as **text**, not as a lambda, because that is what the GUI can store -
a function cannot be saved in a parameter field. NeuroWorkflow compiles the text, with
`np`, `math`, `random`, `statistics` and `functools` available inside it.

In [ ]:
AMPA, NMDA, GABA_A, GABA_B = 1, 2, 3, 4

# --- weights (pA peak). Compare w x tau, not w — see the table above.
w_ampa, w_nmda   =   40.0,  2.0
w_gabaa, w_gabab = -150.0, -8.0
w_drive          =  140.0

def pathway(n):
    """Connectivity for pathway number n: 10% sparse, like Brunel, but repeatable.

    The random generator is seeded with the pathway number and the pair of node ids,
    so asking the same question twice gives the same answer. That is what lets the
    AMPA and NMDA projections of one pathway choose the SAME pairs - they are two
    components of one synapse, not two independent synapses. A plain np.random.rand()
    would draw afresh for each projection and scatter them.

    Written as text rather than a lambda because the GUI stores parameters as values;
    NeuroWorkflow compiles the text, and np is available inside it.
    """
    return ("lambda src, tgt: 1 if np.random.default_rng("
            f"[{n}, src['node_id'], tgt['node_id']]).random() < 0.1 else 0")

neuron = dict(
    model_type     = "point_neuron",
    model_template = "nest:iaf_psc_alpha_multisynapse",
    location       = "VISp",
    layer          = "L4",
    nest_params = {
        "C_m":     250.0,   # pF
        "tau_m":   10.0,    # ms
        "t_ref":   2.0,     # ms
        "V_th":    -55.0,   # mV
        "V_reset": -70.0,   # mV
        "E_L":     -70.0,   # mV
        # one time constant per receptor: AMPA, NMDA, GABA_A, GABA_B
        "tau_syn": [2.0, 100.0, 6.0, 200.0],
    },
)

exc.configure(pop_name="exc", N=80, ei_type="exc", **neuron)
inh.configure(pop_name="inh", N=20, ei_type="inh", **neuron)

# External drive: 100 independent Poisson trains, arriving on AMPA.
drive.configure(
    pop_name       = "ext",
    n_trains       = 100,
    distribution   = "poisson",
    firing_rate_hz = 40.0,
    start_ms       = 1.0,
    stop_ms        = 1000.0,   # must cover the whole run
    seed           = 1,
)


def projection(source, target, receptor, weight, name, rule):
    """One receptor's share of a pathway. 'name' must be unique: it becomes the
    synapse JSON filename, and a repeated name silently overwrites."""
    return {
        "source": source, "target": target,
        "connection_rule":      rule,
        "dynamics_params":      f"{name}.json",
        "dynamics_params_dict": {"receptor_type": receptor},
        "syn_weight":           weight,
    }


conn.configure(
    model_template  = "static_synapse",
    delay           = 1.5,
    allow_autapses  = False,
    allow_multapses = False,
    connections = [
        # external drive -> both populations, fast excitation only
        projection("ext", "exc", AMPA,   w_drive, "ext_exc_ampa",  pathway(0)),
        projection("ext", "inh", AMPA,   w_drive, "ext_inh_ampa",  pathway(1)),

        # excitatory pathways: AMPA + NMDA on the same pairs
        projection("exc", "exc", AMPA,   w_ampa,  "exc_exc_ampa",  pathway(2)),
        projection("exc", "exc", NMDA,   w_nmda,  "exc_exc_nmda",  pathway(2)),
        projection("exc", "inh", AMPA,   w_ampa,  "exc_inh_ampa",  pathway(3)),
        projection("exc", "inh", NMDA,   w_nmda,  "exc_inh_nmda",  pathway(3)),

        # inhibitory pathways: GABA_A + GABA_B on the same pairs
        projection("inh", "exc", GABA_A, w_gabaa, "inh_exc_gabaa", pathway(4)),
        projection("inh", "exc", GABA_B, w_gabab, "inh_exc_gabab", pathway(4)),
        projection("inh", "inh", GABA_A, w_gabaa, "inh_inh_gabaa", pathway(5)),
        projection("inh", "inh", GABA_B, w_gabab, "inh_inh_gabab", pathway(5)),
    ],
)

cfg.configure(
    simulator   = "pointnet",
    config_file = "config_multireceptor.json",
    tstop_ms    = 1000.0,
    dt_ms       = 0.1,
    reports = {
        "v_report": {
            "variable_name": "V_m",
            "cells":         "all",   # the virtual drive population is excluded automatically
            "module":        "membrane_report",
            "sections":      "soma",
        }
    },
)

ana.configure(plot_raster=True, plot_traces=True, plot_rate=True,
              report_name="v_report", rate_bin_ms=10.0, save_figures=True)

## 3. Build and run

In [ ]:
wf = WorkflowBuilder("NW_MultiReceptor_EI_Network")

for node in [exc, inh, drive, conn, cfg, ana]:
    wf.add_node(node)

# Fan-in: all three populations meet at Connectivity
wf.connect("PopExc", "population", "Connectivity", "populations")
wf.connect("PopInh", "population", "Connectivity", "populations")
wf.connect("Drive",  "population", "Connectivity", "populations")

wf.connect("Connectivity", "network", "SimConfig", "populations")
wf.connect("SimConfig",    "results", "Analysis",  "results")

wf.context["results_path"] = "./results/multireceptor"

workflow = wf.build()
ok = workflow.execute()
assert ok, "Workflow failed — check printed errors above"
print("Workflow completed successfully")

## 4. Validate outputs

A balanced network should land in the asynchronous irregular regime: moderate rates and a
CV near 1, meaning the intervals are as variable as a Poisson process. A CV near 0 would
mean clock-like firing, and a rate in the hundreds would mean the inhibition is not
holding.

In [ ]:
print("Output validation:")
for name, node in workflow.nodes.items():
    for pname, port in node._output_ports.items():
        print(f"  {name}.{pname}: {'OK' if port.value is not None else '*** None ***'}")

rates = ana._output_ports["firing_rate_hz"].value
isi   = ana._output_ports["isi_stats"].value
print()
for pop in ("exc", "inh"):
    print(f"  {pop}: {rates[pop]:.2f} Hz, CV {isi[pop]['cv']:.2f}")

## 5. What each slow receptor contributes

The slow receptors carry small weights, which makes it tempting to treat them as a detail.
They are not: charge is `w x tau`, so a 100 ms receptor at weight 2 outweighs a 2 ms
receptor at weight 40.

Each variant below sets one weight to zero and re-runs. Note that each gets its **own**
`results_path`: changing the connections changes the network, and rebuilding one into a
directory already used in this session fails on a file the previous run still holds
open.

In [ ]:
import copy

baseline      = copy.deepcopy(conn._parameters["connections"])
baseline_rate = dict(ana._output_ports["firing_rate_hz"].value)   # read before re-running

def rerun(label, results_path, **weight_overrides):
    connections = copy.deepcopy(baseline)
    for c in connections:
        for name, weight in weight_overrides.items():
            if c["dynamics_params"] == f"{name}.json":
                c["syn_weight"] = weight
    conn.configure(connections=connections)
    cfg.configure(config_file=f"config_{label}.json")
    for node in workflow.nodes.values():
        node._context["results_path"] = results_path
    assert workflow.execute(), f"{label} failed"
    r = ana._output_ports["firing_rate_hz"].value
    print(f"  {label:16} exc {r['exc']:6.2f} Hz   inh {r['inh']:6.2f} Hz")

print(f"  {'all four':16} exc {baseline_rate['exc']:6.2f} Hz   "
      f"inh {baseline_rate['inh']:6.2f} Hz   (the run above)")
rerun("no_nmda",  "./results/multireceptor_no_nmda",
      exc_exc_nmda=0.0, exc_inh_nmda=0.0)
rerun("no_gabab", "./results/multireceptor_no_gabab",
      inh_exc_gabab=0.0, inh_inh_gabab=0.0)

### Reading the result

Removing **NMDA** lowers the rate: the slow excitatory current was providing sustained
depolarization between fast AMPA events, and without it fewer cells reach threshold.

Removing **GABA_B** raises the rate by about half. Slow inhibition is the brake on this
network, and at weight 8 it was contributing more charge than GABA_A at weight 150.

Both effects come from receptors whose weights look negligible next to AMPA's. That is the
practical reason to think in `w x tau` when tuning a multi-receptor model: the weight alone
tells you almost nothing about what a receptor contributes.

### Where to go next

- Give the external drive an NMDA component as well, and watch the network smooth out.
- Make the drive rate a search dimension — `firing_rate_hz` on `NW_SpikeTrains` is
  declared optimizable — and target a firing rate for `exc`.
- Replace the deterministic pathway rule with a random one and compare: AMPA and NMDA
  will then land on different pairs, which is a different model, not a reseeding.